In [1]:
! pip install streamlit pandas numpy networkx scikit-learn matplotlib pytesseract pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 88.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 104.1 MB/s eta 0:00:0000:01


In [4]:
import pandas as pd
import networkx as nx
import numpy as np
from sklearn.ensemble import IsolationForest

# 1. LAYER 1: GRAPH NETWORK (Connections Check)
def check_graph_risk(vendor_id, officer_id, vendors_df, officers_df):
    # Graph banao
    G = nx.Graph()
    
    # Nodes add karo (Logic: Phone number same hai toh link banega)
    for _, row in vendors_df.iterrows():
        G.add_node(row['Vendor_ID'], type='Vendor', phone=row['Phone'])
    for _, row in officers_df.iterrows():
        G.add_node(row['Officer_ID'], type='Officer', linked_phone=row['Linked_Vendor_Phone'])
        
    # Edges add karo (Agar Phone match hua toh Fraud Link)
    v_phone = vendors_df[vendors_df['Vendor_ID'] == vendor_id]['Phone'].values[0]
    off_linked_phone = officers_df[officers_df['Officer_ID'] == officer_id]['Linked_Vendor_Phone'].values[0]
    
    if v_phone == off_linked_phone:
        return "CRITICAL ALERT: Direct Link Found (Shared Phone Number)"
    
    return "Network Check Passed"

# 2. LAYER 2: AI ANOMALY (Amount Check)
def check_anomaly(amount, transactions_df):
    # Model Train (Har baar live train hota hai demo ke liye)
    model = IsolationForest(contamination=0.1)
    X = transactions_df[['Amount']].values
    model.fit(X)
    
    # Predict
    pred = model.predict([[amount]])
    if pred[0] == -1:
        return "HIGH RISK: Amount is an Anomaly (Too High/Low)"
    return "Amount looks Normal"

# 3. LAYER 3: BENFORD'S LAW (Fake Number Check)
def check_benford(amount):
    first_digit = int(str(amount)[0])
    # Benford ke hisaab se 1 se shuru hone wale number 30% hone chahiye
    # Ye simple rule hai: Agar amount 9 se shuru ho raha hai toh shaq hai
    if first_digit >= 8: 
        return "WARNING: High First Digit (Suspicious Pattern)"
    return "Statistical Check Passed"

In [5]:
import pandas as pd
import networkx as nx
import numpy as np
from sklearn.ensemble import IsolationForest
from datetime import datetime

# ==========================================
# 1. LAYER 1: GRAPH NETWORK (Connections)
# ==========================================
def check_network_risk(vendor_id, officer_id, df_vendors, df_officers):
    """
    Checks for Direct and Indirect links (Shared Phone/Address)
    """
    G = nx.Graph()
    
    # Nodes add karo
    for _, row in df_vendors.iterrows():
        G.add_node(row['Vendor_ID'], type='Vendor', phone=row['Phone'], address=row['Address'])
    
    for _, row in df_officers.iterrows():
        G.add_node(row['Officer_ID'], type='Officer', phone=row['Phone'])

    # Edges (Connections) banao based on Shared Info
    # Check 1: Shared Phone
    vendor_phone = df_vendors[df_vendors['Vendor_ID'] == vendor_id]['Phone'].values[0]
    officer_phone = df_officers[df_officers['Officer_ID'] == officer_id]['Phone'].values[0]
    
    if vendor_phone == officer_phone:
        return "🚨 CRITICAL: Direct Collusion Detected (Shared Phone Number)"

    # Check 2: Shared Address (Cartel Check between Vendors)
    # (Demo ke liye hum simple direct check kar rahe hain)
    
    return "✅ Network Scan Passed (No direct links found)"

# ==========================================
# 2. LAYER 2: AI ANOMALY (Isolation Forest)
# ==========================================
def check_ai_anomaly(current_amount, df_transactions):
    """
    Uses Machine Learning to find if the amount is weird (Outlier)
    """
    # Purana data taiyaar karo training ke liye
    X = df_transactions[['Amount']].values
    
    # Model Train karo (Demo ke liye real-time training)
    model = IsolationForest(contamination=0.05, random_state=42)
    model.fit(X)
    
    # Prediction: -1 means Anomaly (Fraud), 1 means Normal
    pred = model.predict([[current_amount]])
    
    if pred[0] == -1:
        return "⚠️ HIGH RISK: AI detected an abnormal amount (Outlier)"
    return "✅ AI Check Passed (Amount within normal range)"

# ==========================================
# 3. LAYER 3: SMURFING & BENFORD (Rules)
# ==========================================
def check_stat_rules(current_amount, vendor_id, df_transactions):
    """
    Checks for Split Transactions (Smurfing) and Fake Numbers
    """
    # Rule 1: Smurfing Check (Kya aaj hi chote-chote payments hue hain?)
    # Filter transactions for this vendor today (Mock logic for demo)
    recent_txns = df_transactions[df_transactions['Vendor_ID'] == vendor_id]
    
    # Agar pichle transactions ka total + current amount limit cross kare
    # (Yahan hum simple count check kar rahe hain demo ke liye)
    if len(recent_txns) > 50: # Agar bohot zyada transactions hain
        return "⚠️ SMURFING ALERT: Too many transactions for this vendor"

    # Rule 2: Benford's Law (Leading Digit Check)
    first_digit = int(str(int(current_amount))[0])
    
    # Usually digits 7, 8, 9 high amounts mein kam hote hain naturally
    # Agar amount starts with 9 (e.g. 99,000 to stay under 1 Lakh limit)
    if first_digit == 9 and current_amount < 100000:
        return "⚠️ PATTERN ALERT: Amount starts with 9 (Possible limit evasion)"
        
    return "✅ Statistical Rules Passed"

# ==========================================
# 4. NEW: SHELL COMPANY CHECK (Age)
# ==========================================
def check_shell_company(vendor_id, df_vendors):
    """
    Checks if company is too new (Shell Company)
    """
    # Agar tumne Registration_Date column add kiya hai toh ye chalega
    if 'Registration_Date' in df_vendors.columns:
        reg_date_str = df_vendors[df_vendors['Vendor_ID'] == vendor_id]['Registration_Date'].values[0]
        reg_date = datetime.strptime(reg_date_str, "%Y-%m-%d").date()
        today = datetime.today().date()
        
        days_old = (today - reg_date).days
        
        if days_old < 30: # Agar company 1 mahine se bhi nayi hai
            return f"🚨 SHELL COMPANY ALERT: Vendor is only {days_old} days old!"
            
    return "✅ Company Age Verified"

# ==========================================
# MASTER FUNCTION (Sabko Run Karega)
# ==========================================
def run_fraud_scan(vendor_id, officer_id, amount, df_ven, df_off, df_txn):
    
    logs = []
    risk_score = 0
    
    # 1. Run Graph Check
    msg1 = check_network_risk(vendor_id, officer_id, df_ven, df_off)
    logs.append(msg1)
    if "CRITICAL" in msg1: risk_score += 50
    
    # 2. Run AI Check
    msg2 = check_ai_anomaly(amount, df_txn)
    logs.append(msg2)
    if "HIGH RISK" in msg2: risk_score += 30
    
    # 3. Run Stats Check
    msg3 = check_stat_rules(amount, vendor_id, df_txn)
    logs.append(msg3)
    if "ALERT" in msg3: risk_score += 20
    
    # 4. Run Shell Company Check
    msg4 = check_shell_company(vendor_id, df_ven)
    logs.append(msg4)
    if "SHELL" in msg4: risk_score += 40
    
    # Final Decision
    final_status = "APPROVED"
    if risk_score >= 50:
        final_status = "BLOCKED"
        
    return final_status, risk_score, logs

In [ ]:
import streamlit as st
import pandas as pd
import logic  # Hamari logic file import ki

# Page Setup
st.set_page_config(page_title="FiscalNet AI", layout="wide")
st.title("🛡️ FiscalNet: Pre-Payment Fraud Blocker")

# Sidebar: Simulate Input
st.sidebar.header("📝 Upload Invoice Data")
vendor_id = st.sidebar.selectbox("Select Vendor", ["V001", "V002", "V003", "V004", "V005"])
officer_id = st.sidebar.selectbox("Select Approving Officer", ["OFF01", "OFF02", "OFF03"])
invoice_amount = st.sidebar.number_input("Invoice Amount", min_value=1000, value=50000)

if st.sidebar.button("🔍 Scan & Verify"):
    # Load Data
    ven_df = pd.read_csv("data/vendors.csv")
    off_df = pd.read_csv("data/officers.csv")
    txn_df = pd.read_csv("data/transactions.csv")
    
    # ---------------- RUN FRAUD SCAN ----------------
    final_status, risk_score, logs = logic.run_fraud_scan(vendor_id, officer_id, invoice_amount, ven_df, off_df, txn_df)
    
    # ---------------- DISPLAY RESULTS ----------------
    col1, col2, col3, col4 = st.columns(4)
    
    # Layer 1: Network
    with col1:
        st.subheader("🕸️ Network Graph")
        if "CRITICAL" in logs[0]:
            st.error(logs[0])
        else:
            st.success(logs[0])

    # Layer 2: AI Anomaly
    with col2:
        st.subheader("🤖 AI Anomaly")
        if "HIGH RISK" in logs[1]:
            st.error(logs[1])
        else:
            st.success(logs[1])
            
    # Layer 3: Statistical Rules
    with col3:
        st.subheader("📊 Statistical")
        if "ALERT" in logs[2]:
            st.warning(logs[2])
        else:
            st.success(logs[2])
            
    # Layer 4: Shell Company
    with col4:
        st.subheader("🏢 Company Age")
        if "SHELL" in logs[3]:
            st.error(logs[3])
        else:
            st.success(logs[3])
            
    # ---------------- RISK SCORE ----------------
    st.markdown("---")
    st.subheader(f"🔥 Risk Score: {risk_score}/100")
    
    # ---------------- FINAL VERDICT ----------------
    if final_status == "BLOCKED":
        st.header("🚨 FINAL VERDICT: PAYMENT BLOCKED")
        st.error("Fraud Detected. Do not process this bill.")
    else:
        st.header("✅ FINAL VERDICT: APPROVED")
        st.success("Safe to proceed.")

ModuleNotFoundError: No module named 'logic'